### Audio Speech Sentiment Analysis using CNN + BiLSTM + Attention
A deep learning project for analyzing and classifying speech audio. Utilizes advanced audio feature extraction (MFCCs, spectral features) combined with a temporal deep learning model to deliver accurate and insightful audio analysis.

In [ ]:
!pip uninstall -y resampy librosa
!pip install resampy librosa


In [ ]:
# Basic Imports
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Music Imports
import librosa
import librosa.display(for visual spectogram and waveform)

# Keras Imports
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Scikit Learn Imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Load Dataset


In [2]:
train_csv = pd.read_csv('TRAIN.csv')
train_audio_path = 'TRAIN'
test_audio_path = 'TEST'

# Data Preprocessing

In [3]:
def extract_features(file_name):
    audio, sample_rate = librosa.load(file_name, res_type='kaiser_fast')
    mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    return np.mean(mfccs.T, axis=0)

features = []
labels = []

for index, row in train_csv.iterrows():
    file_path = os.path.join(train_audio_path, row['Filename'])
    data = extract_features(file_path)
    features.append(data)
    labels.append(row['Class'])

In [ ]:
# # View after processing all data
# print(f"\n--- Final Data (Last Audio Processed) ---\n{data}")
# print(f"\n--- All Features (List) ---\n{features}")
# print(f"\n--- All Labels (List) ---\n{labels}")

# Audio EDA

In [ ]:
# Example Audio Visualization
example_audio = os.path.join(train_audio_path, train_csv['Filename'].iloc[0])
audio, sr = librosa.load(example_audio)

# Waveform
plt.figure(figsize=(14, 4))
librosa.display.waveshow(audio, sr=sr)
plt.title('Waveform of Example Audio')
plt.show()

# Spectrogram
plt.figure(figsize=(10, 4))
spectrogram = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
librosa.display.specshow(spectrogram, sr=sr, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram of Example Audio')
plt.show()


In [ ]:
#Spectrogram Comparison Across Classes
plt.figure(figsize=(12,8))

for i, cls in enumerate(classes):
    sample_file = train_csv[train_csv['Class']==cls]['Filename'].sample(1).values[0]
    audio_path = os.path.join(train_audio_path, sample_file)
    audio, sr = librosa.load(audio_path)
    S = librosa.stft(audio)
    S_db = librosa.amplitude_to_db(abs(S))

    plt.subplot(3,1,i+1)
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f"Spectrogram - Class: {cls}")

plt.tight_layout()
plt.show()


In [ ]:
#MFCC Heatmap for Random Samples
import random
import librosa.display

# Pick one random sample from each class
classes = train_csv['Class'].unique()
plt.figure(figsize=(12, 8))

for i, cls in enumerate(classes):
    sample_file = train_csv[train_csv['Class']==cls]['Filename'].sample(1).values[0]
    audio_path = os.path.join(train_audio_path, sample_file)
    audio, sr = librosa.load(audio_path)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    
    plt.subplot(3,1,i+1)
    librosa.display.specshow(mfccs, x_axis='time')
    plt.colorbar()
    plt.title(f"MFCC Heatmap - Class: {cls}")

plt.tight_layout()
plt.show()


In [ ]:
#Audio Feature Comparison: Spectral Centroid vs Zero-Crossing Rate
import librosa
import numpy as np

# Pick one random sample per class
plt.figure(figsize=(12, 8))

for i, cls in enumerate(classes):
    sample_file = train_csv[train_csv['Class']==cls]['Filename'].sample(1).values[0]
    audio_path = os.path.join(train_audio_path, sample_file)
    
    audio, sr = librosa.load(audio_path)
    
    # Spectral Centroid
    spec_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    # Zero-Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y=audio)[0]
    
    # Plot
    plt.subplot(3,1,i+1)
    plt.plot(spec_centroid, color='r', label='Spectral Centroid')
    plt.plot(zcr * max(spec_centroid), color='b', alpha=0.7, label='Zero Crossing Rate (scaled)')
    plt.title(f"Spectral Centroid & ZCR - Class: {cls}")
    plt.xlabel("Frames")
    plt.legend()

plt.tight_layout()
plt.show()


# Train Test Split

In [5]:
X = np.array(features)
le = LabelEncoder()
y = to_categorical(le.fit_transform(labels))

# Train-Test Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Building CNN+BiLSTM+Attention For Audio Sentiment

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM
from tensorflow.keras.layers import Dense, Dropout, Layer, BatchNormalization
import tensorflow as tf

# ---- Attention Layer ----
class Attention(Layer):
    def __init__(self):
        super(Attention, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight",
                                 shape=(input_shape[-1], 1),
                                 initializer="normal")

        self.b = self.add_weight(name="att_bias",
                                 shape=(input_shape[1], 1),
                                 initializer="zeros")

    def call(self, x):
        # Alignment scores
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        # Softmax to get weights
        a = tf.keras.backend.softmax(e, axis=1)
        # Weighted average
        output = x * a
        return tf.keras.backend.sum(output, axis=1)

# ---- Reshape for 1D ----
X_train = X_train.reshape(X_train.shape[0], 40, 1)
X_val = X_val.reshape(X_val.shape[0], 40, 1)

# ---- Model ----
model = Sequential()

# CNN Block 1
model.add(Conv1D(64, kernel_size=3, padding='same', activation='relu', input_shape=(40,1)))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

# CNN Block 2
model.add(Conv1D(128, kernel_size=3, padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

# BiLSTM Block
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.3))

# Attention Layer
model.add(Attention())

# Output
model.add(Dense(3, activation='softmax'))

# Compile
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32
)


# Evaluation

In [ ]:
# Evaluate the Model
loss, accuracy = model.evaluate(X_val, y_val)
print(f'Validation Loss: {loss:.4f}')
print(f'Validation Accuracy: {accuracy:.4f}')

# Predict Classes
y_pred = model.predict(X_val)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_val, axis=1)

# Classification Report 
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, target_names=le.classes_))

# Confusion Matrix 
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()


# Prediction System

In [17]:
import numpy as np
import librosa
from IPython.display import Audio  # For playing audio

# Detection System
def predict_sentiment(audio_path):
    feature = extract_features(audio_path)         # Extract MFCC features
    feature = feature.reshape(1, 40, 1)         # Reshape for CNN input
    prediction = model.predict(feature)            # Predict sentiment
    predicted_label = le.inverse_transform([np.argmax(prediction)])[0]
    return predicted_label

# Save Files

In [ ]:
import joblib

# Save the trained model
model.save('sentiment_cnn_model.h5')

# Save the Label Encoder
joblib.dump(le, 'label_encoder.pkl')
